# Day 5-01｜投籃分析 Pipeline 摘要
> Python 籃球運動資料分析課程  
> 讀取 Day 4 的姿態與球軌跡結果，整理成展示用摘要指標。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 讀取 pose angle 與 ball track 輸出。
- 計算核心摘要指標。
- 輸出 JSON 供成果展示使用。

## 完成產出
- `d5_01_shooting_summary.json` 分析摘要檔。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 尋找 Day 4 結果檔。
2. 計算摘要數值。
3. 寫出成果 JSON。


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

bootstrap_candidates = [
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
]
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，確認課程已同步到 /content/drive/MyDrive/basketball_hackathon/course。"
    )

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT, mount_drive=False)


In [ ]:
import json
import pandas as pd
from src.shooting_utils import estimate_release_frame

RESULTS = COURSE_ROOT / "assets" / "results"
pose_csv = RESULTS / "d4_02_pose_angles.csv"
ball_csv = RESULTS / "d4_03_ball_track.csv"

if not pose_csv.exists():
    raise FileNotFoundError(f"找不到姿態結果：{pose_csv}。請先完成 Day 4-02。")
if not ball_csv.exists():
    raise FileNotFoundError(f"找不到球軌跡結果：{ball_csv}。請先完成 Day 4-03。")

pose_df = pd.read_csv(pose_csv)
ball_df = pd.read_csv(ball_csv)

if pose_df.empty:
    raise RuntimeError("d4_02_pose_angles.csv 是空的，請重新執行 Day 4-02。")
if ball_df.empty:
    raise RuntimeError("d4_03_ball_track.csv 是空的，請重新執行 Day 4-03。")

pose_df.head()


In [ ]:
summary = {
    "pose_frames": int(len(pose_df)),
    "max_elbow_angle": float(pose_df["elbow_angle"].max()),
    "min_elbow_angle": float(pose_df["elbow_angle"].min()),
    "max_knee_angle": float(pose_df["knee_angle"].max()),
    "ball_points": int(len(ball_df)),
    "estimated_release_frame": estimate_release_frame(ball_df),
}
summary


In [ ]:
out_json = RESULTS / "d5_01_shooting_summary.json"
out_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", out_json)

這份 summary 不是正式運動科學評估，只是讓學生知道如何把影像分析轉成可報告的數字。